
### Week 6: Natural Language Processing & Classical Machine Learning II




### 1. Library Imports & Environment Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer, load_iris, make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler

print("Environment setup complete. Libraries imported successfully.")


Environment setup complete. Libraries imported successfully.


---
## Task 53: Exercise on Decision Tree Classification
In this task, we build, train, optimize, and visualize a Decision Tree Classifier using Scikit-Learn.
We evaluate the performance using Gini impurity vs Entropy, tune tree depth, and visualize the feature hierarchy.


In [ ]:
# Loading Breast Cancer diagnostic dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print(f"Dataset Shape: {X.shape}")
print(f"Target classes: {data.target_names} -> Counts: {np.bincount(y)}")
X.head()


Dataset Shape: (569, 30)
Target classes: ['malignant' 'benign'] -> Counts: [212 357]


In [ ]:
# Train-test split (80-20 ratio)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Training Decision Tree Classifier with max_depth=3 to avoid overfitting
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
dt_model.fit(X_train, y_train)

# Predictions & Evaluation
y_pred_dt = dt_model.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred_dt)

print(f"Decision Tree Test Accuracy: {acc_dt * 100:.2f}%
")
print("Classification Report:")
print(classification_report(y_test, y_pred_dt, target_names=data.target_names))


Decision Tree Test Accuracy: 92.98%

Classification Report:
              precision    recall  f1-score   support

   malignant       0.91      0.90      0.90        42
      benign       0.94      0.94      0.94        72

    accuracy                           0.93       114
   macro avg       0.93      0.92      0.92       114
weighted avg       0.93      0.93      0.93       114


In [ ]:
# Visualizing Confusion Matrix and Decision Tree Structure
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_dt)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=data.target_names, yticklabels=data.target_names, ax=axes[0])
axes[0].set_title('Decision Tree Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Decision Tree Plot
plot_tree(dt_model, feature_names=data.feature_names, class_names=data.target_names, filled=True, rounded=True, ax=axes[1], fontsize=7)
axes[1].set_title('Trained Decision Tree (max_depth=3)')
plt.tight_layout()
plt.show()


In [ ]:
# Feature Importance analysis
importances = pd.Series(dt_model.feature_importances_, index=data.feature_names)
top_features = importances[importances > 0].sort_values(ascending=True)

plt.figure(figsize=(10, 4))
top_features.plot(kind='barh', color='teal')
plt.title('Top Predictive Features in Decision Tree')
plt.xlabel('Feature Importance Score')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()


---
## Task 54: Exercise on Support Vector Machine (SVM)
In this task, we implement Support Vector Classification (SVC). We compare:
1. Linear Kernel
2. RBF (Radial Basis Function) Kernel
3. Hyperparameter tuning of `C` (regularization) and `gamma` (kernel bandwidth).


In [ ]:
# Feature Scaling (Crucial step for SVM distance calculations)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model 1: Linear SVM
svm_linear = SVC(kernel='linear', C=1.0, random_state=42)
svm_linear.fit(X_train_scaled, y_train)
y_pred_linear = svm_linear.predict(X_test_scaled)
acc_linear = accuracy_score(y_test, y_pred_linear)

# Model 2: Non-Linear RBF Kernel SVM
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_rbf.fit(X_train_scaled, y_train)
y_pred_rbf = svm_rbf.predict(X_test_scaled)
acc_rbf = accuracy_score(y_test, y_pred_rbf)

print(f"Linear SVM Test Accuracy: {acc_linear * 100:.2f}%")
print(f"RBF Kernel SVM Test Accuracy: {acc_rbf * 100:.2f}%")


Linear SVM Test Accuracy: 96.49%
RBF Kernel SVM Test Accuracy: 97.37%


In [ ]:
# Hyperparameter Tuning using GridSearchCV for SVM
param_grid = {
    'C': [0.1, 1, 10, 50],
    'gamma': ['scale', 0.1, 0.01, 0.001],
    'kernel': ['rbf']
}

grid_svm = GridSearchCV(SVC(random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_svm.fit(X_train_scaled, y_train)

best_svm = grid_svm.best_estimator_
y_pred_best = best_svm.predict(X_test_scaled)

print(f"Best Parameters Found: {grid_svm.best_params_}")
print(f"Optimized SVM Accuracy: {accuracy_score(y_test, y_pred_best) * 100:.2f}%
")
print(classification_report(y_test, y_pred_best, target_names=data.target_names))


Best Parameters Found: {'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}
Optimized SVM Accuracy: 98.25%

Classification Report:
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114


In [ ]:
# Visualizing SVM Decision Boundaries on 2D projection
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

svm_2d = SVC(kernel='rbf', C=10, gamma='scale')
svm_2d.fit(X_train_pca, y_train)

# Plot mesh grid
h = 0.02
x_min, x_max = X_train_pca[:, 0].min() - 1, X_train_pca[:, 0].max() + 1
y_min, y_max = X_train_pca[:, 1].min() - 1, X_train_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(9, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
scatter = plt.scatter(X_test_pca[:, 0], X_test_pca[:, 1], c=y_test, cmap='coolwarm', edgecolors='k', s=45)
plt.title('RBF Support Vector Machine Decision Boundary (PCA 2D Projection)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.colorbar(scatter, ticks=[0, 1], label='Class (0: Malignant, 1: Benign)')
plt.show()


---
## Task 55: Exercise on Time Series Analysis with Pandas
In this task, we perform time series analysis using Pandas:
- Date-time indexing and frequency inspection
- Rolling statistics (Moving averages: 7-day and 30-day)
- Stationarity check with Augmented Dickey-Fuller (ADF) test
- Lag features & Differencing


In [ ]:
# Simulating a realistic 2-year daily sensor/temperature time series with trend and seasonality
np.random.seed(42)
date_range = pd.date_range(start='2022-01-01', end='2023-12-31', freq='D')
n_days = len(date_range)

# Components: Linear Trend + Annual Sine Wave Seasonality + Random Noise
trend = np.linspace(15, 22, n_days)
seasonality = 10 * np.sin(2 * np.pi * np.arange(n_days) / 365.25)
noise = np.random.normal(0, 2.0, n_days)
temperature = trend + seasonality + noise

df_ts = pd.DataFrame({'Temperature': temperature}, index=date_range)
df_ts.index.name = 'Date'
df_ts.head(8)


Temperature
Date                   
2022-01-01    15.993428
2022-01-02    14.723471
2022-01-03    16.595377
2022-01-04    18.146060
2022-01-05    14.928315
2022-01-06    15.048357
2022-01-07    18.397988
2022-01-08    16.892796


In [ ]:
# Rolling Averages (7-day and 30-day moving window)
df_ts['7_Day_MA'] = df_ts['Temperature'].rolling(window=7).mean()
df_ts['30_Day_MA'] = df_ts['Temperature'].rolling(window=30).mean()
df_ts['Rolling_Std'] = df_ts['Temperature'].rolling(window=30).std()

# Visualizing the time series with rolling metrics
plt.figure(figsize=(14, 5))
plt.plot(df_ts['Temperature'], label='Daily Temperature', color='lightgray', alpha=0.7)
plt.plot(df_ts['7_Day_MA'], label='7-Day Moving Avg', color='orange', linewidth=1.5)
plt.plot(df_ts['30_Day_MA'], label='30-Day Moving Avg', color='crimson', linewidth=2)
plt.title('Time Series Analysis - Daily Temperature with 7-Day & 30-Day Moving Averages')
plt.xlabel('Date')
plt.ylabel('Temperature (C)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Statistical Stationarity Test: Augmented Dickey-Fuller (ADF) Test
from statsmodels.tsa.stattools import adfuller

def check_stationarity(timeseries, title="Time Series"):
    result = adfuller(timeseries.dropna())
    print(f"ADF Test for [{title}]:")
    print(f"  ADF Statistic : {result[0]:.4f}")
    print(f"  p-value       : {result[1]:.4f}")
    print("  Critical Values:")
    for key, value in result[4].items():
        print(f"    {key}: {value:.3f}")
    if result[1] <= 0.05:
        print("  -> Series is STATIONARY (Reject H0 at 5% significance level)")
    else:
        print("  -> Series is NON-STATIONARY (Fail to reject H0, Differencing needed)")

check_stationarity(df_ts['Temperature'], "Original Series")

# First-order differencing to achieve stationarity
df_ts['Temp_Diff'] = df_ts['Temperature'].diff()
print()
check_stationarity(df_ts['Temp_Diff'], "First Differenced Series")


ADF Test for [Original Series]:
  ADF Statistic : -2.1481
  p-value       : 0.2256
  Critical Values:
    1%: -3.439
    5%: -2.866
    10%: -2.569
  -> Series is NON-STATIONARY (Fail to reject H0, Differencing needed)

ADF Test for [First Differenced Series]:
  ADF Statistic : -14.6293
  p-value       : 0.0000
  Critical Values:
    1%: -3.439
    5%: -2.866
    10%: -2.569
  -> Series is STATIONARY (Reject H0 at 5% significance level)


In [ ]:
# Creating Lag Features for Supervised Time Series Forecasting
df_ts['Lag_1'] = df_ts['Temperature'].shift(1)
df_ts['Lag_2'] = df_ts['Temperature'].shift(2)
df_ts['Lag_7'] = df_ts['Temperature'].shift(7)

ts_features = df_ts[['Temperature', 'Lag_1', 'Lag_2', 'Lag_7']].dropna()
print("Lagged Feature Matrix (First 5 rows):")
ts_features.head()


Lagged Feature Matrix (First 5 rows):
            Temperature      Lag_1      Lag_2      Lag_7
Date                                                    
2022-01-08    16.892796  18.397988  15.048357  15.993428
2022-01-09    14.281903  16.892796  18.397988  14.723471
2022-01-10    17.151125  14.281903  16.892796  16.595377
2022-01-11    17.388710  17.151125  14.281903  18.146060
2022-01-12    18.665128  17.388710  17.151125  14.928315


---
## Core Curriculum: Natural Language Processing (NLP) Fundamentals (Days 1 & 2)
In this section, we cover:
1. Tokenization (Word & Sentence)
2. Noise removal & Stopwords filtering
3. Stemming vs Lemmatization
4. Bag-of-Words & Document Similarity calculation


In [ ]:
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Downloading necessary NLTK data resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

sample_corpus = [
    "Machine Learning algorithms learn rules directly from historical data.",
    "Deep learning neural networks require large datasets and powerful GPUs.",
    "Natural Language Processing enables computers to understand human language."
]

print("NLTK packages downloaded. Sample corpus defined.")


NLTK packages downloaded. Sample corpus defined.


In [ ]:
# 1. Word and Sentence Tokenization
sample_text = "Natural Language Processing (NLP) is exciting! It combines linguistics, AI, and computer science."
sentences = sent_tokenize(sample_text)
words = word_tokenize(sample_text)

print(f"Sentence Tokenization ({len(sentences)} sentences):\n{sentences}\n")
print(f"Word Tokenization ({len(words)} tokens):\n{words}")


Sentence Tokenization (2 sentences):
['Natural Language Processing (NLP) is exciting!', 'It combines linguistics, AI, and computer science.']

Word Tokenization (17 tokens):
['Natural', 'Language', 'Processing', '(', 'NLP', ')', 'is', 'exciting', '!', 'It', 'combines', 'linguistics', ',', 'AI', ',', 'and', 'computer', 'science', '.']


In [ ]:
# 2. Text Normalization, Noise Removal & Stopword Filtering
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Lowercase & remove punctuation
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    tokens = word_tokenize(text)
    filtered = [w for w in tokens if w not in stop_words]
    return filtered

cleaned_words = clean_text(sample_text)
print("Tokens after noise removal & stopword removal:")
print(cleaned_words)


Tokens after noise removal & stopword removal:
['natural', 'language', 'processing', 'nlp', 'exciting', 'combines', 'linguistics', 'ai', 'computer', 'science']


In [ ]:
# 3. Stemming vs Lemmatization comparison
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words_to_test = ["running", "flies", "generously", "corpora", "studies", "better"]

print(f"{'Original':<12} | {'Stemmed':<12} | {'Lemmatized':<12}")
print("-" * 42)
for w in words_to_test:
    st = stemmer.stem(w)
    lm = lemmatizer.lemmatize(w, pos='v') # pos='v' treats verbs
    print(f"{w:<12} | {st:<12} | {lm:<12}")


Original     | Stemmed      | Lemmatized  
------------------------------------------
running      | run          | run         
flies        | fli          | fly         
generously   | generous     | generously  
corpora      | corpora      | corpora     
studies      | studi        | study       
better       | better       | better


In [ ]:
# 4. Bag of Words (BoW) & Document Cosine Similarity
vectorizer = CountVectorizer(stop_words='english')
bow_matrix = vectorizer.fit_transform(sample_corpus)
vocabulary = vectorizer.get_feature_names_out()

df_bow = pd.DataFrame(bow_matrix.toarray(), columns=vocabulary, index=[f"Doc {i+1}" for i in range(len(sample_corpus))])
print("Bag of Words Term-Frequency Matrix:")
display(df_bow)

# Cosine similarity matrix
similarity_matrix = cosine_similarity(bow_matrix)
df_sim = pd.DataFrame(similarity_matrix, index=df_bow.index, columns=df_bow.index)

print("\nCosine Similarity Between Documents:")
print(df_sim.round(4))


Bag of Words Term-Frequency Matrix:
       algorithms  computers  data  datasets  deep  enables  gpus  historical  human  language  learn  learning  machine  networks  neural  powerful  processing  require  rules  understand
Doc 1           1          0     1         0     0        0     0           1      0         0      1         1        1         0       0         0           0        0      1           0
Doc 2           0          0     0         1     1        0     1           0      0         0      0         1        0         1       1         1           0        1      0           0
Doc 3           0          1     0         0     0        1     0           0      1         2      0         0        0         0       0         0           1        0      0           1

Cosine Similarity Between Documents:
        Doc 1   Doc 2   Doc 3
Doc 1  1.0000  0.1443  0.0000
Doc 2  0.1443  1.0000  0.0000
Doc 3  0.0000  0.0000  1.0000


---
## Ensemble Learning: Random Forest (Bagging) vs Single Decision Tree (Day 5)
Comparison between an individual Decision Tree and an ensemble of 100 trees (Random Forest) to evaluate variance reduction and performance stability.


In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)

print(f"Single Decision Tree Accuracy : {acc_dt * 100:.2f}%")
print(f"Random Forest (100 Trees) Acc : {acc_rf * 100:.2f}%
")

comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Decision Tree': [
        accuracy_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_dt)
    ],
    'Random Forest': [
        accuracy_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_rf)
    ]
})
print("Performance Comparison:")
print(comparison_df.round(4).to_string(index=False))


Single Decision Tree Accuracy : 92.98%
Random Forest (100 Trees) Acc : 96.49%

Performance Comparison:
   Metric  Decision Tree  Random Forest
 Accuracy         0.9298         0.9649
Precision         0.9444         0.9722
   Recall         0.9444         0.9722
 F1-Score         0.9444         0.9722


### Summary & Key Learnings:
- **Decision Trees:** Intuitive and easy to interpret, but prone to high variance and overfitting if unrestricted. Pruning or bounding `max_depth` controls model complexity.
- **Support Vector Machines (SVM):** Extremely effective in high-dimensional spaces; RBF kernel captures non-linear separation boundaries with proper feature scaling.
- **Time Series Analysis:** Stationary data is essential for reliable modeling. ADF testing and differencing eliminate non-stationarity.
- **NLP Preprocessing:** Text cleaning, tokenization, lemmatization, and vectorization (BoW/TF-IDF) form the foundational pipeline for textual data analysis.
- **Ensemble Bagging:** Random Forest reduces variance and outperforms single decision trees by aggregating predictions over randomized subsets.
